In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8
tz = 'Asia/Shanghai'

# helper to convert epoch(s) to Asia/Shanghai datetime strings
def epoch_to_shanghai_series(s):
    return pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

def short_list_repr(items, limit=20):
    items = list(items)
    if len(items) == 0:
        return ""
    if len(items) <= limit:
        return ", ".join(map(str, items))
    else:
        shown = ", ".join(map(str, items[:limit]))
        return f"{shown}, ...(+{len(items)-limit} more)"

# load CSV files
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# prepare summaries
summaries = []

# metric.csv summary
m_count = metric_df.shape[0]
m_min_ts = int(metric_df['timestamp'].min()) if m_count>0 else None
m_max_ts = int(metric_df['timestamp'].max()) if m_count>0 else None
m_min_ts_str = epoch_to_shanghai_series(pd.Series([m_min_ts]))[0] if m_min_ts is not None else ""
m_max_ts_str = epoch_to_shanghai_series(pd.Series([m_max_ts]))[0] if m_max_ts is not None else ""
m_cmdb_ids = short_list_repr(metric_df['cmdb_id'].unique()[:20])
m_kpis = short_list_repr(metric_df['kpi_name'].unique()[:20])
sample_metric = metric_df[['timestamp','cmdb_id','kpi_name','value']].copy().head(5)
if not sample_metric.empty:
    sample_metric['timestamp'] = epoch_to_shanghai_series(sample_metric['timestamp'])

summaries.append({
    'file_name': 'metric.csv',
    'row_count': m_count,
    'min_timestamp': m_min_ts_str,
    'max_timestamp': m_max_ts_str,
    'unique_cmdb_ids': m_cmdb_ids,
    'unique_kpi/trace/log_names': m_kpis
})

# trace.csv summary
t_count = trace_df.shape[0]
t_min_ts = int(trace_df['timestamp'].min()) if t_count>0 else None
t_max_ts = int(trace_df['timestamp'].max()) if t_count>0 else None
t_min_ts_str = epoch_to_shanghai_series(pd.Series([t_min_ts]))[0] if t_min_ts is not None else ""
t_max_ts_str = epoch_to_shanghai_series(pd.Series([t_max_ts]))[0] if t_max_ts is not None else ""
t_cmdb_ids = short_list_repr(trace_df['cmdb_id'].unique()[:20])
t_names = short_list_repr(trace_df['trace_name'].unique()[:20])
sample_trace = trace_df[['timestamp','cmdb_id','trace_name','value']].copy().head(5)
if not sample_trace.empty:
    sample_trace['timestamp'] = epoch_to_shanghai_series(sample_trace['timestamp'])

summaries.append({
    'file_name': 'trace.csv',
    'row_count': t_count,
    'min_timestamp': t_min_ts_str,
    'max_timestamp': t_max_ts_str,
    'unique_cmdb_ids': t_cmdb_ids,
    'unique_kpi/trace/log_names': t_names
})

# log.csv summary
l_count = log_df.shape[0]
l_min_ts = int(log_df['timestamp'].min()) if l_count>0 else None
l_max_ts = int(log_df['timestamp'].max()) if l_count>0 else None
l_min_ts_str = epoch_to_shanghai_series(pd.Series([l_min_ts]))[0] if l_min_ts is not None else ""
l_max_ts_str = epoch_to_shanghai_series(pd.Series([l_max_ts]))[0] if l_max_ts is not None else ""
l_cmdb_ids = short_list_repr(log_df['cmdb_id'].unique()[:20])
l_names = short_list_repr(log_df['log_name'].unique()[:20])
sample_log = log_df[['timestamp','cmdb_id','log_name','value']].copy().head(5)
if not sample_log.empty:
    sample_log['timestamp'] = epoch_to_shanghai_series(sample_log['timestamp'])

summaries.append({
    'file_name': 'log.csv',
    'row_count': l_count,
    'min_timestamp': l_min_ts_str,
    'max_timestamp': l_max_ts_str,
    'unique_cmdb_ids': l_cmdb_ids,
    'unique_kpi/trace/log_names': l_names
})

# error_logs.csv summary
e_count = error_df.shape[0]
e_min_ts = int(error_df['timestamp'].min()) if e_count>0 else None
e_max_ts = int(error_df['timestamp'].max()) if e_count>0 else None
e_min_ts_str = epoch_to_shanghai_series(pd.Series([e_min_ts]))[0] if e_min_ts is not None else ""
e_max_ts_str = epoch_to_shanghai_series(pd.Series([e_max_ts]))[0] if e_max_ts is not None else ""
e_cmdb_ids = short_list_repr(error_df['cmdb_id'].unique()[:20])
# error_logs has no KPI name column
e_names = ""
sample_error = error_df[['timestamp','cmdb_id','message']].copy().head(5)
if not sample_error.empty:
    sample_error['timestamp'] = epoch_to_shanghai_series(sample_error['timestamp'])

summaries.append({
    'file_name': 'error_logs.csv',
    'row_count': e_count,
    'min_timestamp': e_min_ts_str,
    'max_timestamp': e_max_ts_str,
    'unique_cmdb_ids': e_cmdb_ids,
    'unique_kpi/trace/log_names': e_names
})

# Create a compact summary DataFrame
summary_df = pd.DataFrame(summaries)[['file_name','row_count','min_timestamp','max_timestamp','unique_cmdb_ids','unique_kpi/trace/log_names']]

# Final outputs: compact summary and up to-5-row samples for each file
summary_df, sample_metric, sample_trace, sample_log, sample_error

```
Out[1]:
```
# Plain English summary of the execution results
result = (
    "Summary of loaded telemetry files (times shown in Asia/Shanghai, CST):\n\n"
    "1) metric.csv: 322,448 rows, time range 2022-03-21 07:00:00 CST to 2022-03-21 08:00:00 CST. "
    "Up to 20 cmdb_id values were captured (examples: adservice, adservice-0, adservice-1, ...). "
    "Sample metric rows include entries such as adservice | app.grpc.count = 69, app.grpc.mrt = 0.002528, app.grpc.rr = 100, app.grpc.sr = 100, app.http.count = 24.\n\n"
    "2) trace.csv: 27,912 rows, time range 2022-03-21 07:30:00 CST to 2022-03-21 07:59:00 CST. "
    "Examples of cmdb_id values include adservice-0, adservice-1, adservice-2, etc. "
    "Sample trace rows include trace.from_frontend-0.duration_mean, trace.from_frontend-0.duration_p95, trace.from_frontend-0.error_rate, trace.from_frontend-0.row_count, trace.from_frontend-1.duration_mean.\n\n"
    "3) log.csv: 1,794 rows, time range 2022-03-21 07:30:00 CST to 2022-03-21 08:00:00 CST. "
    "Examples of cmdb_id values include adservice-0, adservice-1, adservice-2, etc. "
    "Log name examples: log.error_count, log.row_count. Sample log rows show log.error_count = 0 and log.row_count values (e.g., adservice-0 log.row_count = 68).\n\n"
    "4) error_logs.csv: 0 rows (no error log entries found in the file).\n\n"
    "All outputs shown were compact summaries and top-5 samples per file; full raw data was not displayed."
)
result

The original code execution output of IPython Kernel is also provided below for reference:

(        file_name  row_count            min_timestamp            max_timestamp                                    unique_cmdb_ids                         unique_kpi/trace/log_names
0      metric.csv     322448  2022-03-21 07:00:00 CST  2022-03-21 08:00:00 CST  adservice, adservice-0, adservice-1, adservice...  app.grpc.count, app.grpc.mrt, app.grpc.rr, app...
1       trace.csv      27912  2022-03-21 07:30:00 CST  2022-03-21 07:59:00 CST  adservice-0, adservice-1, adservice-2, cartser...  trace.from_frontend-0.duration_mean, trace.fro...
2         log.csv       1794  2022-03-21 07:30:00 CST  2022-03-21 08:00:00 CST  adservice-0, adservice-1, adservice-2, cartser...                     log.error_count, log.row_count
3  error_logs.csv          0                                                                                                                                                        ,                  timestamp    cmdb_id        kpi_name       value
0  2022-03-21 07:00:00 CST  adservice  app.grpc.count   69.000000
1  2022-03-21 07:00:00 CST  adservice    app.grpc.mrt    0.002528
2  2022-03-21 07:00:00 CST  adservice     app.grpc.rr  100.000000
3  2022-03-21 07:00:00 CST  adservice     app.grpc.sr  100.000000
4  2022-03-21 07:00:00 CST  adservice  app.http.count   24.000000,                  timestamp      cmdb_id                           trace_name      value
0  2022-03-21 07:30:00 CST  adservice-0  trace.from_frontend-0.duration_mean   0.000017
1  2022-03-21 07:30:00 CST  adservice-0   trace.from_frontend-0.duration_p95   0.000022
2  2022-03-21 07:30:00 CST  adservice-0     trace.from_frontend-0.error_rate   0.000000
3  2022-03-21 07:30:00 CST  adservice-0      trace.from_frontend-0.row_count  13.000000
4  2022-03-21 07:30:00 CST  adservice-0  trace.from_frontend-1.duration_mean   0.000024,                  timestamp      cmdb_id         log_name  value
0  2022-03-21 07:30:00 CST  adservice-0  log.error_count    0.0
1  2022-03-21 07:30:00 CST  adservice-0    log.row_count   68.0
2  2022-03-21 07:30:00 CST  adservice-1  log.error_count    0.0
3  2022-03-21 07:30:00 CST  adservice-1    log.row_count   50.0
4  2022-03-21 07:30:00 CST  adservice-2  log.error_count    0.0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# ensure metric_df is loaded (reuse if present)
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# timezone
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the entire metric_df (before any filtering)
# ensure timestamp and value types
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

p95_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_P95'})
)

# 2) Convert timestamps to Asia/Shanghai tz-aware datetimes and filter to the incident window (inclusive)
metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

start = pd.Timestamp('2022-03-21 07:30:00', tz=tz)
end = pd.Timestamp('2022-03-21 08:00:00', tz=tz)

metric_window = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] <= end)].copy()

# 3) For each (cmdb_id, kpi_name) compute aggregates over the filtered window.
# Merge global P95 into the windowed dataframe
metric_window = metric_window.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# flag anomalies where value >= global_P95
metric_window['is_anom'] = metric_window['value'] >= metric_window['global_P95']

# aggregates: total_points_in_window, anomaly_count, max_value_in_window
agg_basic = (
    metric_window
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        total_points_in_window=('value', 'size'),
        anomaly_count=('is_anom', 'sum'),
        max_value_in_window=('value', 'max')
    )
)

# earliest_anomaly_timestamp per group (tz-aware). Use min dt where is_anom True.
earliest_anom = (
    metric_window.loc[metric_window['is_anom']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['dt']
    .min()
    .rename(columns={'dt': 'earliest_anomaly_timestamp'})
)

# merge earliest_anom into agg_basic
result_df = agg_basic.merge(earliest_anom, on=['cmdb_id', 'kpi_name'], how='left')

# attach global_P95 to result_df
result_df = result_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# format earliest_anomaly_timestamp as string in Asia/Shanghai; keep NaT as None/empty
result_df['earliest_anomaly_timestamp'] = result_df['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
result_df['earliest_anomaly_timestamp'] = result_df['earliest_anomaly_timestamp'].where(result_df['earliest_anomaly_timestamp'].notna(), None)

# 4) Return compact table limited to top 20 rows ordered by anomaly_count desc, then max_value_in_window desc.
result_df = result_df.sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False])

final_cols = ['cmdb_id', 'kpi_name', 'global_P95', 'total_points_in_window', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']
result_top20 = result_df[final_cols].head(20)

# 5) counts: number of distinct series processed in the window, and number of series that had anomaly_count > 0
num_series_processed = result_df.shape[0]
num_series_with_anom = int((result_df['anomaly_count'] > 0).sum())

# Keep p95_df and metric_window in kernel for later steps (already assigned).
# Output compact results: top 20 rows, and the two counts.
result_top20, num_series_processed, num_series_with_anom

```
Out[2]:
```
summary = (
    "Summary of metric.csv analysis (times in Asia/Shanghai CST):\n\n"
    "- Global P95 was computed per (cmdb_id, kpi_name) using the full timeseries.\n"
    "- Incident window: 2022-03-21 07:30:00 through 2022-03-21 08:00:00 CST.\n"
    "- Series processed: 5,298 distinct (cmdb_id, kpi_name) series. "
    "Series with anomalies (value >= global_P95): 5,118.\n\n"
    "- Top findings (examples from top-ranked rows): many series have anomaly_count = 31 "
    "(an anomaly at every point in the 31-minute window) with earliest anomaly at 2022-03-21 07:30:00 CST. "
    "These include metrics like runtime.java_lang_Runtime_StartTime, container.node-*.container_start_time_seconds, "
    "and system.disk.total. For these series max_value_in_window equals the global P95 (indicating stable/high values), "
    "so the detected 'anomalies' reflect high/static metrics (timestamps or capacity) rather than transient spikes.\n\n"
    "- Conclusion: Large number of series flagged because their values are consistently at/above their global P95 (many are start-time or capacity metrics). "
    "Focus on dynamic performance KPIs for actionable anomalies."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                      cmdb_id                                           kpi_name    global_P95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
857                adservice2                runtime.java_lang_Runtime_StartTime  1.647819e+12                      31             31    2022-03-21 07:30:00 CST         1.647819e+12
250                 adservice                runtime.java_lang_Runtime_StartTime  1.647144e+12                      31             31    2022-03-21 07:30:00 CST         1.647144e+12
3684                   node-5                                  system.disk.total  1.854945e+10                      31             31    2022-03-21 07:30:00 CST         1.854945e+10
3450                   node-1                                  system.disk.total  1.406688e+10                      31             31    2022-03-21 07:30:00 CST         1.406688e+10
3509                   node-2                                  system.disk.total  1.406688e+10                      31             31    2022-03-21 07:30:00 CST         1.406688e+10
3567                   node-3                                  system.disk.total  1.406688e+10                      31             31    2022-03-21 07:30:00 CST         1.406688e+10
238                 adservice  runtime.java_lang_OperatingSystem_CommittedVir...  3.591926e+09                      31             31    2022-03-21 07:30:00 CST         3.591926e+09
1003             adservice2-0      container.node-6.container_start_time_seconds  1.647819e+09                      31             31    2022-03-21 07:30:00 CST         1.647819e+09
1405           cartservice2-0      container.node-5.container_start_time_seconds  1.647782e+09                      31             31    2022-03-21 07:30:00 CST         1.647782e+09
1109            cartservice-0      container.node-6.container_start_time_seconds  1.647760e+09                      31             31    2022-03-21 07:30:00 CST         1.647760e+09
1305            cartservice-2      container.node-6.container_start_time_seconds  1.647759e+09                      31             31    2022-03-21 07:30:00 CST         1.647759e+09
1209            cartservice-1      container.node-6.container_start_time_seconds  1.647757e+09                      31             31    2022-03-21 07:30:00 CST         1.647757e+09
3993         paymentservice-2      container.node-6.container_start_time_seconds  1.647318e+09                      31             31    2022-03-21 07:30:00 CST         1.647318e+09
2333       currencyservice2-0      container.node-6.container_start_time_seconds  1.647236e+09                      31             31    2022-03-21 07:30:00 CST         1.647236e+09
2245        currencyservice-2      container.node-6.container_start_time_seconds  1.647236e+09                      31             31    2022-03-21 07:30:00 CST         1.647236e+09
3109               frontend-2      container.node-5.container_start_time_seconds  1.647144e+09                      31             31    2022-03-21 07:30:00 CST         1.647144e+09
4801  recommendationservice-2      container.node-6.container_start_time_seconds  1.647144e+09                      31             31    2022-03-21 07:30:00 CST         1.647144e+09
1793        checkoutservice-2      container.node-5.container_start_time_seconds  1.647144e+09                      31             31    2022-03-21 07:30:00 CST         1.647144e+09
2921               frontend-1      container.node-5.container_start_time_seconds  1.647144e+09                      31             31    2022-03-21 07:30:00 CST         1.647144e+09
5189        shippingservice-2      container.node-5.container_start_time_seconds  1.647144e+09                      31             31    2022-03-21 07:30:00 CST         1.647144e+09, 5298, 5118)```
```

In [ ]:
# In[3]:

import re
from collections import Counter
import pandas as pd

# reuse result_df (per-series aggregates) from previous steps
# patterns to exclude (case-insensitive substring match)
exclude_patterns = [
    'start_time',
    'container_start_time',
    'runtime.java_lang',
    'system.disk.total',
    'disk.total',
    'uptime',
    'capacity',
    'node_boot_time'
]

# build mask: True for rows to keep (i.e., not matching any exclude pattern)
mask_keep = ~result_df['kpi_name'].str.lower().apply(
    lambda name: any(pat in name for pat in exclude_patterns)
)

# apply mask and keep only series with anomaly_count > 0
filtered = result_df[mask_keep].copy()
filtered = filtered[filtered['anomaly_count'] > 0]

# order and select final columns
final_cols = ['cmdb_id', 'kpi_name', 'global_P95', 'total_points_in_window',
              'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']

filtered_sorted = filtered.sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False])
filtered_top20 = filtered_sorted[final_cols].head(20)

# count of series after filtering
num_after_filter = int(filtered.shape[0])

# extract tokens from kpi_name to suggest dynamic KPI families
def tokenize(name):
    # split on dots, underscores, dashes, slashes, and other non-alphanum chars
    toks = re.split(r'[^A-Za-z0-9\*]+', name.lower())
    # keep meaningful tokens (non-empty, not purely numeric)
    return [t for t in toks if t and not t.isnumeric()]

all_names = filtered['kpi_name'].unique()
counter = Counter()
for n in all_names:
    counter.update(set(tokenize(n)))  # use set to count token presence per kpi_name once

top_tokens_df = (
    pd.DataFrame(counter.items(), columns=['token', 'count'])
      .sort_values(by='count', ascending=False)
      .head(10)
      .reset_index(drop=True)
)

# Return compact outputs
filtered_top20, num_after_filter, top_tokens_df

```
Out[3]:
```
summary = (
    "Filtered results (excluded static/start-time and disk/capacity metrics) — plain summary:\n\n"
    "- Total series after filtering (and keeping anomaly_count > 0): 4,601.\n\n"
    "- Top examples (from top 20): many container memory metrics (e.g., container.node-6.container_memory_rss and container.node-6.container_memory_cache) "
    "across services like adservice-*, emailservice-*, recommendationservice-*, paymentservice-*. These series show anomaly_count = 31 (anomalous at every minute in the 31-minute window), "
    "with earliest anomaly at 2022-03-21 07:30:00 CST and max values equal to their global P95 (i.e., steady/high values rather than transient spikes).\n\n"
    "- Top 10 kpi_name tokens observed (to guide investigation of dynamic KPI families):\n"
    "  1) istio (872)\n"
    "  2) mesh (872)\n"
    "  3) grpc (584)\n"
    "  4) source (488)\n"
    "  5) request (416)\n"
    "  6) destination (384)\n"
    "  7) bytes (376)\n"
    "  8) response (239)\n"
    "  9) frontend2 (238)\n"
    " 10) frontend (218)\n\n"
    "Interpretation: after removing static/start-time and disk/capacity metrics, many container memory series remain flagged because they are consistently at or above their global P95 during the window. "
    "There is also a strong presence of istio/mesh/grpc/request-related metrics — these dynamic families (app.grpc.*, app.http.*, istio/mesh network metrics, cpu/memory) are good next targets for investigation."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                       cmdb_id                                 kpi_name   global_P95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
377                adservice-0    container.node-6.container_memory_rss  231456768.0                      31             31    2022-03-21 07:30:00 CST          231456768.0
557                adservice-2    container.node-5.container_memory_rss  148381696.0                      31             31    2022-03-21 07:30:00 CST          148381696.0
471                adservice-1    container.node-6.container_memory_rss  139186176.0                      31             31    2022-03-21 07:30:00 CST          139186176.0
1386            cartservice2-0    container.node-5.container_memory_rss   82448384.0                      31             31    2022-03-21 07:30:00 CST           82448384.0
976               adservice2-0  container.node-6.container_memory_cache   68530176.0                      31             31    2022-03-21 07:30:00 CST           68530176.0
2554            emailservice-2    container.node-6.container_memory_rss   39632896.0                      31             31    2022-03-21 07:30:00 CST           39632896.0
4782   recommendationservice-2    container.node-6.container_memory_rss   39612416.0                      31             31    2022-03-21 07:30:00 CST           39612416.0
4684   recommendationservice-1    container.node-6.container_memory_rss   39387136.0                      31             31    2022-03-21 07:30:00 CST           39387136.0
2632           emailservice2-0    container.node-6.container_memory_rss   38232064.0                      31             31    2022-03-21 07:30:00 CST           38232064.0
4880  recommendationservice2-0    container.node-6.container_memory_rss   37806080.0                      31             31    2022-03-21 07:30:00 CST           37806080.0
2480            emailservice-1    container.node-6.container_memory_rss   37748736.0                      31             31    2022-03-21 07:30:00 CST           37748736.0
2402            emailservice-0    container.node-6.container_memory_rss   37273600.0                      31             31    2022-03-21 07:30:00 CST           37273600.0
4586   recommendationservice-0    container.node-6.container_memory_rss   37154816.0                      31             31    2022-03-21 07:30:00 CST           37154816.0
4676   recommendationservice-1  container.node-6.container_memory_cache   35504128.0                      31             31    2022-03-21 07:30:00 CST           35504128.0
2472            emailservice-1  container.node-6.container_memory_cache   34893824.0                      31             31    2022-03-21 07:30:00 CST           34893824.0
2394            emailservice-0  container.node-6.container_memory_cache   34750464.0                      31             31    2022-03-21 07:30:00 CST           34750464.0
2138         currencyservice-1  container.node-6.container_memory_cache   34553856.0                      31             31    2022-03-21 07:30:00 CST           34553856.0
2546            emailservice-2  container.node-6.container_memory_cache   34504704.0                      31             31    2022-03-21 07:30:00 CST           34504704.0
3974          paymentservice-2    container.node-6.container_memory_rss   34131968.0                      31             31    2022-03-21 07:30:00 CST           34131968.0
3834          paymentservice-0    container.node-6.container_memory_rss   33873920.0                      31             31    2022-03-21 07:30:00 CST           33873920.0, 4601,          token  count
0        istio    872
1         mesh    872
2         grpc    584
3       source    488
4      request    416
5  destination    384
6        bytes    376
7     response    239
8    frontend2    238
9     frontend    218)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse filtered DataFrame from previous steps (contains only series with anomaly_count>0 and excluded patterns removed)
# ensure tz is set
tz = pytz.timezone('Asia/Shanghai')

# 1) generate token = first two dot-separated segments of kpi_name
def make_token(name):
    parts = str(name).split('.')
    if len(parts) == 1:
        return parts[0]
    else:
        return parts[0] + '.' + parts[1]

filtered['token'] = filtered['kpi_name'].map(make_token)

# 2) prepare earliest_anomaly timestamp as tz-aware datetime for accurate min
def parse_earliest(ts_str):
    if pd.isna(ts_str) or ts_str is None:
        return pd.NaT
    # drop trailing timezone token (e.g., "CST") and localize to Asia/Shanghai
    base = str(ts_str).rsplit(' ', 1)[0]
    try:
        return pd.to_datetime(base).tz_localize(tz)
    except Exception:
        return pd.NaT

filtered['earliest_dt'] = filtered['earliest_anomaly_timestamp'].map(parse_earliest)

# group by token and compute aggregates
grouped = filtered.groupby('token', as_index=False).agg(
    series_count=('kpi_name', 'nunique'),
    total_anomaly_points=('anomaly_count', 'sum'),
    earliest_anom_dt=('earliest_dt', 'min'),
    representative_kpi_names=('kpi_name', lambda s: list(pd.Index(s).unique()[:5]))
)

# series_with_anomaly_count equals series_count here since filtered had anomaly_count>0 for all series,
# but include explicitly for clarity
grouped['series_with_anomaly_count'] = grouped['series_count']

# format earliest_anom_dt to string (Asia/Shanghai)
def fmt_dt(dt):
    if pd.isna(dt):
        return None
    return dt.strftime('%Y-%m-%d %H:%M:%S %Z')

grouped['earliest_anomaly_timestamp'] = grouped['earliest_anom_dt'].map(fmt_dt)

# select desired columns and sort
token_agg_df = grouped[['token', 'series_count', 'total_anomaly_points', 'earliest_anomaly_timestamp', 'representative_kpi_names']].copy()
token_agg_df = token_agg_df.sort_values(by='total_anomaly_points', ascending=False).reset_index(drop=True)

# top 20 tokens and total token count
token_top20 = token_agg_df.head(20)
num_tokens = int(token_agg_df.shape[0])

# Keep token_agg_df in kernel for next steps
token_top20, num_tokens

```
Out[4]:
```
Summary of token-level aggregates (based on the filtered series with anomaly_count > 0):

- Total distinct tokens found: 44.

- Top tokens by total_anomaly_points (showing token — series_count — total_anomaly_points — earliest anomaly):
  1) container.node-6 — 63 series — 53,952 points — earliest 2022-03-21 07:30:00 CST  
     (representative KPIs: container.node-6.container_memory_rss, container.node-6.container_memory_cache, ...)
  2) mesh.source — 488 series — 19,244 points — earliest 2022-03-21 07:30:00 CST  
     (representative KPIs: mesh.source.adservice2.*.istio_tcp_*, mesh.source.*.istio_* ...)
  3) mesh.destination — 384 series — 14,455 points — earliest 2022-03-21 07:30:00 CST
  4) container.node-5 — 63 series — 9,767 points — earliest 2022-03-21 07:30:00 CST
  5) system.swap — 6 series — 1,116 points — earliest 2022-03-21 07:30:00 CST
  6) system.net — 14 series — 1,037 points — earliest 2022-03-21 07:30:00 CST
  7) app.grpc — 4 series — 615 points — earliest 2022-03-21 07:30:00 CST
  8) system.io — 9 series — 543 points — earliest 2022-03-21 07:30:00 CST
  (top-20 table was produced and saved; these are the highest-impact tokens)

- Interpretation: the largest anomaly volume comes from container memory metrics on node-6 and service-to-service mesh metrics (istio/mesh source & destination). Other notable dynamic families: system swap, network (system.net), app.grpc, system.io, and several JVM/runtime groups.

This token-level view points you to investigate:
- container memory usage on node-6 and node-5 (many container.memory_rss/cache series),
- Istio/mesh traffic paths (mesh.source / mesh.destination),
- network, swap, and I/O metrics,
- grpc and http request-related KPIs for application-level issues.

The original code execution output of IPython Kernel is also provided below for reference:

(                                           token  series_count  total_anomaly_points earliest_anomaly_timestamp                           representative_kpi_names
0                               container.node-6            63                 53952    2022-03-21 07:30:00 CST  [container.node-6.container_memory_rss, contai...
1                                    mesh.source           488                 19244    2022-03-21 07:30:00 CST  [mesh.source.adservice2.basic-tidb.istio_tcp_s...
2                               mesh.destination           384                 14455    2022-03-21 07:30:00 CST  [mesh.destination.unknown.adservice2.istio_res...
3                               container.node-5            63                  9767    2022-03-21 07:30:00 CST  [container.node-5.container_memory_rss, contai...
4                                    system.swap             6                  1116    2022-03-21 07:30:00 CST  [system.swap.free, system.swap.si, system.swap...
5                                     system.net            14                  1037    2022-03-21 07:30:00 CST  [system.net.packets_in.error, system.net.packe...
6                                       app.grpc             4                   615    2022-03-21 07:30:00 CST  [app.grpc.rr, app.grpc.sr, app.grpc.count, app...
7                                      system.io             9                   543    2022-03-21 07:30:00 CST  [system.io.rkb_s, system.io.r_s, system.io.r_a...
8                 runtime.jvm_memory_pool_MB_max             6                   372    2022-03-21 07:30:00 CST  [runtime.jvm_memory_pool_MB_max.Compressed_Cla...
9                runtime.jvm_memory_pool_MB_init             6                   372    2022-03-21 07:30:00 CST  [runtime.jvm_memory_pool_MB_init.Tenured_Gen, ...
10  runtime.jvm_memory_pool_allocated_MB_created             6                   372    2022-03-21 07:30:00 CST  [runtime.jvm_memory_pool_allocated_MB_created....
11                     runtime.jvm_threads_state             6                   312    2022-03-21 07:30:00 CST  [runtime.jvm_threads_state.WAITING, runtime.jv...
12                                   system.disk             3                   291    2022-03-21 07:30:00 CST  [system.disk.pct_usage, system.disk.readonly, ...
13          runtime.jvm_memory_pool_MB_committed             6                   279    2022-03-21 07:30:00 CST  [runtime.jvm_memory_pool_MB_committed.Tenured_...
14                                     system.os             3                   243    2022-03-21 07:30:00 CST  [system.os.nofile.max, system.os.nofile.used_p...
15                                    system.mem             7                   236    2022-03-21 07:30:00 CST  [system.mem.total, system.mem.real.pct_useage,...
16                                    system.tcp             1                   186    2022-03-21 07:30:00 CST                           [system.tcp.retrans_pct]
17                                    system.udp             1                   186    2022-03-21 07:30:00 CST                           [system.udp.connect.num]
18                                     system.fs             2                   184    2022-03-21 07:30:00 CST   [system.fs.inodes.in_use, system.fs.inodes.used]
19                                      app.http             4                   164    2022-03-21 07:30:00 CST  [app.http.rr, app.http.sr, app.http.count, app..., 44)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# reuse trace_df if present, else load
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")

# ensure types
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# timezone (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, trace_name) using the full trace_df (global thresholds before filtering)
trace_p95_df = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_P95'})
)

# 2) Convert timestamps to Asia/Shanghai tz-aware datetimes and filter to the incident window (inclusive)
trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start = pd.Timestamp('2022-03-21 07:30:00', tz=tz)
end = pd.Timestamp('2022-03-21 08:00:00', tz=tz)
trace_window = trace_df[(trace_df['dt'] >= start) & (trace_df['dt'] <= end)].copy()

# 3) Merge global P95 into the windowed dataframe and flag anomalies (value >= global_P95)
trace_window = trace_window.merge(trace_p95_df, on=['cmdb_id', 'trace_name'], how='left')
trace_window['is_anom'] = trace_window['value'] >= trace_window['global_P95']

# Aggregates per (cmdb_id, trace_name) over the filtered window
agg_basic_trace = (
    trace_window
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        total_points_in_window=('value', 'size'),
        anomaly_count=('is_anom', 'sum'),
        max_value_in_window=('value', 'max')
    )
)

# earliest_anomaly_timestamp per group (first dt where is_anom True)
earliest_trace_anom = (
    trace_window.loc[trace_window['is_anom']]
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['dt']
    .min()
    .rename(columns={'dt': 'earliest_anomaly_timestamp'})
)

# merge earliest timestamp and global_P95
result_trace = agg_basic_trace.merge(earliest_trace_anom, on=['cmdb_id', 'trace_name'], how='left')
result_trace = result_trace.merge(trace_p95_df, on=['cmdb_id', 'trace_name'], how='left')

# format earliest_anomaly_timestamp as string in Asia/Shanghai; keep NaT as None
result_trace['earliest_anomaly_timestamp'] = result_trace['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
result_trace['earliest_anomaly_timestamp'] = result_trace['earliest_anomaly_timestamp'].where(result_trace['earliest_anomaly_timestamp'].notna(), None)

# 4) Compact table: top 20 rows ordered by anomaly_count desc then max_value_in_window desc
result_trace = result_trace.sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False])
trace_cols = ['cmdb_id', 'trace_name', 'global_P95', 'total_points_in_window', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']
result_trace_top20 = result_trace[trace_cols].head(20)

# 5) counts: total distinct series processed (in the window) and how many had anomaly_count > 0
num_trace_series_processed = int(result_trace.shape[0])
num_trace_series_with_anom = int((result_trace['anomaly_count'] > 0).sum())

# Keep trace_p95_df and trace_window saved in kernel for later steps (they are defined).
# Final compact outputs:
result_trace_top20, num_trace_series_processed, num_trace_series_with_anom

```
Out[5]:
```
Summary of trace-level anomaly detection (times in Asia/Shanghai CST):

- What I did: computed global P95 per (cmdb_id, trace_name) from the full trace.csv, filtered to the incident window 2022-03-21 07:30:00–08:00:00 CST, and for each series computed total points, anomaly_count (value >= global_P95), earliest anomaly time, and max value in the window.

- Counts: 1,080 distinct (cmdb_id, trace_name) series were processed in the window; 1,080 series had anomaly_count > 0.

- Top observations (examples from the top-ranked rows):
  - Many trace series (e.g., trace.from_frontend-*.error_rate, trace.self.duration_p95, trace.self.error_rate) have global_P95 = 0.0 and were flagged as anomalous for every point in the window (anomaly_count = 30, earliest anomaly = 2022-03-21 07:30:00 CST, max_value_in_window = 0.0).
  - The window contains 30 points per series (07:30 through 07:59), and series with constant zero-valued metrics get flagged because their values equal the global P95 of 0.0.

- Interpretation / recommendation:
  - Many detected “anomalies” reflect metrics that are constant at or below zero (global P95 = 0) rather than meaningful performance spikes. These are likely not actionable.
  - To find actionable trace anomalies, focus on series where global_P95 > 0 (non-zero baselines) or where values show transient spikes above historical P95 (e.g., duration or latency metrics with non-zero P95), rather than metrics that are zero/stable across time.

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id                        trace_name  global_P95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
2      adservice-0  trace.from_frontend-0.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
6      adservice-0  trace.from_frontend-1.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
10     adservice-0  trace.from_frontend-2.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
14     adservice-1  trace.from_frontend-0.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
18     adservice-1  trace.from_frontend-1.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
22     adservice-1  trace.from_frontend-2.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
26     adservice-2  trace.from_frontend-0.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
30     adservice-2  trace.from_frontend-1.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
34     adservice-2  trace.from_frontend-2.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
50   cartservice-0  trace.from_frontend-0.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
54   cartservice-0  trace.from_frontend-1.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
58   cartservice-0  trace.from_frontend-2.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
61   cartservice-0           trace.self.duration_p95         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
62   cartservice-0             trace.self.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
78   cartservice-1  trace.from_frontend-0.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
82   cartservice-1  trace.from_frontend-1.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
86   cartservice-1  trace.from_frontend-2.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
89   cartservice-1           trace.self.duration_p95         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
90   cartservice-1             trace.self.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0
106  cartservice-2  trace.from_frontend-0.error_rate         0.0                      30             30    2022-03-21 07:30:00 CST                  0.0, 1080, 1080)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# reuse filtered per-series aggregates (must exist from prior steps)
if 'filtered' not in globals():
    raise RuntimeError("Expected 'filtered' DataFrame in kernel (per-series aggregates).")

tz = pytz.timezone('Asia/Shanghai')

# ensure helper columns exist
if 'earliest_dt' not in filtered.columns:
    # parse earliest_anomaly_timestamp to tz-aware datetime if needed
    def parse_earliest(ts_str):
        if pd.isna(ts_str) or ts_str is None:
            return pd.NaT
        base = str(ts_str).rsplit(' ', 1)[0]
        try:
            return pd.to_datetime(base).tz_localize(tz)
        except Exception:
            return pd.NaT
    filtered['earliest_dt'] = filtered['earliest_anomaly_timestamp'].map(parse_earliest)

# mark series that are always anomalous in the window
filtered['always_anom'] = (filtered['anomaly_count'] == filtered['total_points_in_window'])

# select series where kpi_name contains 'container.node-6' (case-insensitive)
mask_node6 = filtered['kpi_name'].str.contains('container.node-6', case=False, na=False)
node6_series = filtered[mask_node6].copy()

# group by cmdb_id and compute aggregates
grouped_node6 = node6_series.groupby('cmdb_id', as_index=False).agg(
    series_count=('kpi_name', 'nunique'),
    total_anomaly_points=('anomaly_count', 'sum'),
    earliest_anom_dt=('earliest_dt', 'min'),
    always_anomalous_series_count=('always_anom', 'sum'),
    representative_kpi_names=('kpi_name', lambda s: list(pd.Index(s).unique()[:5]))
)

# format earliest_anom_dt to string in Asia/Shanghai timezone
def fmt_dt(dt):
    if pd.isna(dt):
        return None
    return dt.strftime('%Y-%m-%d %H:%M:%S %Z')

grouped_node6['earliest_anomaly_timestamp'] = grouped_node6['earliest_anom_dt'].map(fmt_dt)

# select and order columns as requested
out_cols = ['cmdb_id', 'series_count', 'total_anomaly_points', 'always_anomalous_series_count',
            'earliest_anomaly_timestamp', 'representative_kpi_names']

grouped_node6 = grouped_node6[out_cols]

# sort and limit to top 20 rows
node6_df_top20 = grouped_node6.sort_values(by=['series_count', 'total_anomaly_points'], ascending=[False, False]).head(20).reset_index(drop=True)

# total number of distinct cmdb_id values found on node-6
num_cmdb_node6 = int(grouped_node6['cmdb_id'].nunique())

# final compact outputs
node6_df_top20, num_cmdb_node6

```
Out[6]:
```
Summary of node-6 series (case-insensitive match 'container.node-6'):

- Number of distinct cmdb_id values found on node-6: 36.

- Top cmdb_id rows (ordered by series_count then total_anomaly_points) — examples:
  - paymentservice2-0: 63 series, 1,810 total anomaly points, 58 series always anomalous, earliest anomaly 2022-03-21 07:30:00 CST (representative KPIs: container.node-6.container_memory_rss, container_memory_cache, ...)
  - shippingservice2-0: 63 series, 1,727 points, 53 always-anomalous series, earliest 07:30:00 CST
  - frontend2-0: 63 series, 1,721 points, 53 always-anomalous series, earliest 07:30:00 CST
  - checkoutservice2-0, recommendationservice2-0, shippingservice-1, frontend-0, checkoutservice-1, paymentservice-1, checkoutservice-0, ... (see top-20 for full list)

- Key observations:
  - Most cmdb_ids have 62–63 distinct container.node-6 kpi series detected.
  - A large fraction of those series are "always anomalous" (anomaly_count == total_points_in_window), i.e., anomalous at every minute in the 31-minute window. Earliest anomaly for virtually all is 2022-03-21 07:30:00 CST.
  - Representative KPI names are dominated by container memory metrics (container_memory_rss, container_memory_cache, mapped_file, etc.), indicating persistent/high container memory values on node-6 across many pods/services.

- Interpretation / next steps:
  - The signals point to widespread persistent container memory values on node-6 (many pods on that node show values at/above their historical P95 during the incident window). Investigate node-6 resource state (host memory usage, OOMs, cgroup limits) and pod placement/resource limits for the listed services (especially those with high always_anomalous_series_count).

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  series_count  total_anomaly_points  always_anomalous_series_count earliest_anomaly_timestamp                           representative_kpi_names
0          paymentservice2-0            63                  1810                             58    2022-03-21 07:30:00 CST  [container.node-6.container_memory_rss, contai...
1         shippingservice2-0            63                  1727                             53    2022-03-21 07:30:00 CST  [container.node-6.container_memory_rss, contai...
2                frontend2-0            63                  1721                             53    2022-03-21 07:30:00 CST  [container.node-6.container_memory_rss, contai...
3         checkoutservice2-0            63                  1703                             54    2022-03-21 07:30:00 CST  [container.node-6.container_memory_rss, contai...
4   recommendationservice2-0            63                  1696                             54    2022-03-21 07:30:00 CST  [container.node-6.container_memory_rss, contai...
5          shippingservice-1            63                  1653                             52    2022-03-21 07:30:00 CST  [container.node-6.container_memory_cache, cont...
6                 frontend-0            63                  1612                             51    2022-03-21 07:30:00 CST  [container.node-6.container_memory_rss, contai...
7          checkoutservice-1            63                  1607                             49    2022-03-21 07:30:00 CST  [container.node-6.container_memory_rss, contai...
8           paymentservice-1            63                  1607                             49    2022-03-21 07:30:00 CST  [container.node-6.container_memory_rss, contai...
9          checkoutservice-0            63                  1606                             49    2022-03-21 07:30:00 CST  [container.node-6.container_memory_rss, contai...
10         currencyservice-1            63                  1531                             47    2022-03-21 07:30:00 CST  [container.node-6.container_memory_cache, cont...
11               adservice-1            63                  1513                             47    2022-03-21 07:30:00 CST  [container.node-6.container_memory_rss, contai...
12             cartservice-2            63                  1513                             47    2022-03-21 07:30:00 CST  [container.node-6.container_memory_cache, cont...
13             cartservice-1            63                  1472                             46    2022-03-21 07:30:00 CST  [container.node-6.container_memory_cache, cont...
14   productcatalogservice-0            63                  1409                             43    2022-03-21 07:30:00 CST  [container.node-6.container_memory_cache, cont...
15         currencyservice-0            63                  1377                             41    2022-03-21 07:30:00 CST  [container.node-6.container_memory_mapped_file...
16  productcatalogservice2-0            62                  1710                             54    2022-03-21 07:30:00 CST  [container.node-6.container_memory_cache, cont...
17   productcatalogservice-1            62                  1648                             51    2022-03-21 07:30:00 CST  [container.node-6.container_memory_rss, contai...
18            emailservice-2            62                  1632                             49    2022-03-21 07:30:00 CST  [container.node-6.container_memory_rss, contai...
19   productcatalogservice-2            62                  1603                             50    2022-03-21 07:30:00 CST  [container.node-6.container_memory_cache, cont..., 36)```
```

In [ ]:
# In[7]:

# reuse existing result_trace from prior steps (per-(cmdb_id, trace_name) aggregates)
# Filter trace series with global_P95 > 0 and anomaly_count > 0, then return compact top-20 and the count.

# ensure required variable exists
if 'result_trace' not in globals():
    raise RuntimeError("Expected 'result_trace' DataFrame in kernel (trace aggregates).")

# define columns to return (reuse if present)
trace_cols = ['cmdb_id', 'trace_name', 'global_P95', 'total_points_in_window',
              'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']

# filter
trace_anom_pos = result_trace[(result_trace['global_P95'] > 0) & (result_trace['anomaly_count'] > 0)].copy()

# sort and select top 20
trace_anom_top20 = trace_anom_pos.sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False])[trace_cols].head(20)

# total count meeting the criteria
num_trace_anom_pos = int(trace_anom_pos.shape[0])

# outputs (DataFrame and count)
trace_anom_top20, num_trace_anom_pos

```
Out[7]:
```
Summary of trace-series with global_P95 > 0 and anomaly_count > 0:

- What I filtered: trace series where global P95 > 0 and there was at least one point in the incident window (2022-03-21 07:30–08:00 CST) with value >= that P95.

- Counts: 771 distinct (cmdb_id, trace_name) series meet the criteria.

- Top observations (examples from the top-ranked rows):
  - Many of the flagged traces are row_count metrics for inter-service calls, e.g.:
    - checkoutservice-0 / trace.from_frontend-1.row_count — global_P95 = 1.0, 21 points in-window, anomaly_count = 21, earliest anomaly 2022-03-21 07:30:00 CST, max = 1.0
    - frontend-1 / trace.to_checkoutservice-0.row_count — global_P95 = 1.0, 21 points, anomaly_count = 21, earliest 07:30:00 CST, max = 1.0
    - Multiple checkoutservice/*, frontend/*, paymentservice/*, emailservice/* traces appear repeatedly with anomaly_count equal to most or all points in-window (e.g., 21, 20, 18, 17).
  - Earliest anomalies for these series are at the start of the window (mostly 2022-03-21 07:30:00 or 07:31:00 CST).
  - Max values in-window equal the global P95 for many series (e.g., 1.0), indicating the values are at or just at the historical 95th-percentile during the window rather than transient spikes.

- Interpretation / next steps:
  - The contributing traces concentrate on request row_count between frontend → checkout → payment/email paths. Investigate this service chain (frontend, checkoutservice, paymentservice, emailservice) for traffic/behavior changes during the window.
  - Because many series have values exactly at P95 (and anomaly_count equals nearly all points), prioritize traces with non-trivial max > global_P95 or those showing sudden increases in duration/latency rather than stable row_count==P95, to find actionable performance regressions.

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                              trace_name  global_P95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
127  checkoutservice-0         trace.from_frontend-1.row_count         1.0                      21             21    2022-03-21 07:30:00 CST                  1.0
311  checkoutservice-2         trace.from_frontend-1.row_count         1.0                      21             21    2022-03-21 07:30:00 CST                  1.0
631         frontend-1    trace.to_checkoutservice-0.row_count         1.0                      21             21    2022-03-21 07:30:00 CST                  1.0
639         frontend-1    trace.to_checkoutservice-2.row_count         1.0                      21             21    2022-03-21 07:30:00 CST                  1.0
307  checkoutservice-2         trace.from_frontend-0.row_count         1.0                      20             20    2022-03-21 07:31:00 CST                  1.0
547         frontend-0    trace.to_checkoutservice-2.row_count         1.0                      20             20    2022-03-21 07:31:00 CST                  1.0
271  checkoutservice-1     trace.to_paymentservice-0.row_count         1.0                      18             18    2022-03-21 07:30:00 CST                  1.0
359  checkoutservice-2       trace.to_emailservice-2.row_count         1.0                      18             18    2022-03-21 07:30:00 CST                  1.0
503     emailservice-2  trace.from_checkoutservice-2.row_count         1.0                      18             18    2022-03-21 07:30:00 CST                  1.0
787   paymentservice-0  trace.from_checkoutservice-1.row_count         1.0                      18             18    2022-03-21 07:30:00 CST                  1.0
183  checkoutservice-0     trace.to_paymentservice-1.row_count         1.0                      17             17    2022-03-21 07:31:00 CST                  1.0
267  checkoutservice-1       trace.to_emailservice-2.row_count         1.0                      17             17    2022-03-21 07:30:00 CST                  1.0
275  checkoutservice-1     trace.to_paymentservice-1.row_count         1.0                      17             17    2022-03-21 07:30:00 CST                  1.0
351  checkoutservice-2       trace.to_emailservice-0.row_count         1.0                      17             17    2022-03-21 07:31:00 CST                  1.0
363  checkoutservice-2     trace.to_paymentservice-0.row_count         1.0                      17             17    2022-03-21 07:31:00 CST                  1.0
367  checkoutservice-2     trace.to_paymentservice-1.row_count         1.0                      17             17    2022-03-21 07:30:00 CST                  1.0
479     emailservice-0  trace.from_checkoutservice-2.row_count         1.0                      17             17    2022-03-21 07:31:00 CST                  1.0
499     emailservice-2  trace.from_checkoutservice-1.row_count         1.0                      17             17    2022-03-21 07:30:00 CST                  1.0
791   paymentservice-0  trace.from_checkoutservice-2.row_count         1.0                      17             17    2022-03-21 07:31:00 CST                  1.0
795   paymentservice-1  trace.from_checkoutservice-0.row_count         1.0                      17             17    2022-03-21 07:31:00 CST                  1.0, 771)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# reuse existing log_df if present, else load
if 'log_df' not in globals():
    log_df = pd.read_csv("log.csv")

# ensure types
log_df['timestamp'] = log_df['timestamp'].astype(int)
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# 1) Filter to incident window (inclusive)
log_df['dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start = pd.Timestamp('2022-03-21 07:30:00', tz=tz)
end = pd.Timestamp('2022-03-21 08:00:00', tz=tz)
log_window = log_df[(log_df['dt'] >= start) & (log_df['dt'] <= end)].copy()

# 2) Per (cmdb_id, log_name) aggregates
# total_points_in_window: count of rows in window
# sum_value_in_window: sum of value (NaN treated as skipped)
# max_value_in_window: max value
agg_basic = (
    log_window
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        total_points_in_window=('value', 'size'),
        sum_value_in_window=('value', 'sum'),
        max_value_in_window=('value', 'max')
    )
)

# earliest_nonzero_timestamp: earliest dt where value > 0
earliest_nonzero = (
    log_window.loc[log_window['value'] > 0]
    .groupby(['cmdb_id', 'log_name'], as_index=False)['dt']
    .min()
    .rename(columns={'dt': 'earliest_nonzero_dt'})
)

# merge
log_agg = agg_basic.merge(earliest_nonzero, on=['cmdb_id', 'log_name'], how='left')

# format earliest_nonzero_timestamp as string in Asia/Shanghai; keep None where missing
log_agg['earliest_nonzero_timestamp'] = log_agg['earliest_nonzero_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
log_agg['earliest_nonzero_timestamp'] = log_agg['earliest_nonzero_timestamp'].where(log_agg['earliest_nonzero_timestamp'].notna(), None)

# 3) Top 20 rows where sum_value_in_window > 0 ordered by sum_value_in_window desc
log_top20 = (
    log_agg[log_agg['sum_value_in_window'] > 0]
    .sort_values(by=['sum_value_in_window', 'max_value_in_window'], ascending=[False, False])
    [['cmdb_id', 'log_name', 'total_points_in_window', 'sum_value_in_window', 'max_value_in_window', 'earliest_nonzero_timestamp']]
    .head(20)
    .reset_index(drop=True)
)

# 4) counts: total distinct (cmdb_id, log_name) series processed in window, and how many with sum_value_in_window > 0
num_log_series_processed = int(log_agg.shape[0])
num_log_series_with_nonzero = int((log_agg['sum_value_in_window'] > 0).sum())

# Keep variables in kernel: log_window, log_agg, log_top20, num_log_series_processed, num_log_series_with_nonzero
log_top20, num_log_series_processed, num_log_series_with_nonzero

```
Out[8]:
```
Summary of log.csv analysis for the incident window 2022-03-21 07:30:00–08:00:00 CST:

- Data processed: 62 distinct (cmdb_id, log_name) series in-window.  
- Series with non-zero log activity (sum_value_in_window > 0): 31 series.

Top log series by total log counts (examples from top 20):
- frontend-0 — log.row_count: 31 points, sum = 23,533, max = 946, earliest non-zero = 2022-03-21 07:30:00 CST
- frontend-1 — log.row_count: 31 points, sum = 23,485, max = 954, earliest non-zero = 2022-03-21 07:30:00 CST
- productcatalogservice-2 — log.row_count: 31 points, sum = 13,958, max = 534, earliest = 07:30:00 CST
- productcatalogservice-1 — log.row_count: 30 points, sum = 13,893, max = 528, earliest = 07:30:00 CST
- cartservice-* and currencyservice-* also show high row_count totals (e.g., ~12k for cartservice nodes; ~10.6k for currencyservice nodes)
- Many earliest non-zero timestamps are at the window start (2022-03-21 07:30:00 CST).

Interpretation / next steps:
- The highest log counts are at the frontends, indicating heavy request volume entering the system during the window; downstream services (productcatalog, cartservice, currencyservice, recommendationservice) also show large counts.
- These log.row_count results point to traffic/throughput patterns rather than error spikes (no error_count entries dominated the top list). Investigate request load, any latency or downstream saturation correlating with the high counts (e.g., frontends' max per-minute counts ~900+).

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id       log_name  total_points_in_window  sum_value_in_window  max_value_in_window earliest_nonzero_timestamp
0                frontend-0  log.row_count                      31              23533.0                946.0    2022-03-21 07:30:00 CST
1                frontend-1  log.row_count                      31              23485.0                954.0    2022-03-21 07:30:00 CST
2   productcatalogservice-2  log.row_count                      31              13958.0                534.0    2022-03-21 07:30:00 CST
3   productcatalogservice-1  log.row_count                      30              13893.0                528.0    2022-03-21 07:30:00 CST
4   productcatalogservice-0  log.row_count                      30              13893.0                521.0    2022-03-21 07:30:00 CST
5             cartservice-2  log.row_count                      30              12172.0                459.0    2022-03-21 07:30:00 CST
6             cartservice-1  log.row_count                      30              12164.0                459.0    2022-03-21 07:30:00 CST
7             cartservice-0  log.row_count                      30              12144.0                459.0    2022-03-21 07:30:00 CST
8                frontend-2  log.row_count                      30              11533.0                474.0    2022-03-21 07:30:00 CST
9         currencyservice-1  log.row_count                      30              10616.0                441.0    2022-03-21 07:30:00 CST
10        currencyservice-0  log.row_count                      30              10593.0                439.0    2022-03-21 07:30:00 CST
11        currencyservice-2  log.row_count                      30              10592.0                428.0    2022-03-21 07:30:00 CST
12  recommendationservice-2  log.row_count                      30               3094.0                115.0    2022-03-21 07:30:00 CST
13  recommendationservice-0  log.row_count                      30               3089.0                114.0    2022-03-21 07:30:00 CST
14  recommendationservice-1  log.row_count                      30               3087.0                115.0    2022-03-21 07:30:00 CST
15              adservice-0  log.row_count                      30               1880.0                 80.0    2022-03-21 07:30:00 CST
16              adservice-1  log.row_count                      31               1528.0                 64.0    2022-03-21 07:30:00 CST
17              adservice-2  log.row_count                      30               1526.0                 66.0    2022-03-21 07:30:00 CST
18        shippingservice-2  log.row_count                      30               1370.0                 74.0    2022-03-21 07:30:00 CST
19        shippingservice-1  log.row_count                      30               1370.0                 73.0    2022-03-21 07:30:00 CST, 62, 31)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# reuse metric_df if available, else load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# ensure types
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# 1) select node rows exactly matching node-1 .. node-6
node_list = ['node-1','node-2','node-3','node-4','node-5','node-6']
node_metric_df = metric_df[metric_df['cmdb_id'].isin(node_list)].copy()

# 2) compute global P95 per (cmdb_id, kpi_name) using the full node series (before filtering)
node_p95_df = (
    node_metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_P95'})
)

# 3) filter to incident window (inclusive) and compute window aggregates
# add tz-aware datetime column
node_metric_df['dt'] = pd.to_datetime(node_metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start = pd.Timestamp('2022-03-21 07:30:00', tz=tz)
end = pd.Timestamp('2022-03-21 08:00:00', tz=tz)
node_window = node_metric_df[(node_metric_df['dt'] >= start) & (node_metric_df['dt'] <= end)].copy()

# merge global P95 into windowed data
node_window = node_window.merge(node_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# flag anomalies (value >= global_P95)
node_window['is_anom'] = node_window['value'] >= node_window['global_P95']

# aggregates per series in window
node_agg_basic = (
    node_window
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        total_points_in_window=('value', 'size'),
        anomaly_count=('is_anom', 'sum'),
        max_value_in_window=('value', 'max')
    )
)

# earliest anomaly timestamp where is_anom True
earliest_node_anom = (
    node_window.loc[node_window['is_anom']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['dt']
    .min()
    .rename(columns={'dt': 'earliest_anomaly_dt'})
)

# merge earliest timestamp and global_P95 (ensure global_P95 present)
node_result = node_agg_basic.merge(earliest_node_anom, on=['cmdb_id', 'kpi_name'], how='left')
node_result = node_result.merge(node_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# 3 continued) compute severity_ratio = max_value_in_window / (global_P95 + 1e-12)
eps = 1e-12
node_result['severity_ratio'] = node_result['max_value_in_window'] / (node_result['global_P95'] + eps)

# format earliest_anomaly_timestamp as string in Asia/Shanghai; keep None for missing
node_result['earliest_anomaly_timestamp'] = node_result['earliest_anomaly_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
node_result['earliest_anomaly_timestamp'] = node_result['earliest_anomaly_timestamp'].where(node_result['earliest_anomaly_timestamp'].notna(), None)

# 4) prepare compact table ordered by severity_ratio desc then anomaly_count desc
out_cols = ['cmdb_id', 'kpi_name', 'global_P95', 'total_points_in_window',
            'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio']
node_result_sorted = node_result.sort_values(by=['severity_ratio', 'anomaly_count'], ascending=[False, False])
node_top20 = node_result_sorted[out_cols].head(20).reset_index(drop=True)

# 5) counts:
# (A) total number of distinct (cmdb_id,kpi_name) series processed for these nodes in the window
total_series_processed_nodes = int(node_result.shape[0])
# (B) number of those series with anomaly_count > 0
num_series_with_anom_nodes = int((node_result['anomaly_count'] > 0).sum())

# Save node-level aggregates for follow-up steps:
node_p95_df = node_p95_df  # global P95 thresholds per node series
node_window = node_window  # filtered window rows for node series
node_result = node_result  # per-series window aggregates with severity_ratio
node_top20, total_series_processed_nodes, num_series_with_anom_nodes

```
Out[9]:
```
summary = (
    "Node-level KPI summary (incident window 2022-03-21 07:30–08:00 CST):\n\n"
    "- Data processed: 352 distinct (cmdb_id,kpi_name) node series; 311 of those had anomaly_count > 0.\n\n"
    "- Highest severity_ratio entries are dominated by IO metrics on node-4 and node-5 (e.g., system.io.rkb_s, system.io.r_s, system.io.r_await). "
    "Many of these have global_P95 = 0 so severity_ratio is extremely large — this indicates the window values are non-zero while historical 95th-percentile was zero (IO activity spike).\n\n"
    "- Notable node-level findings:\n"
    "  * node-4 and node-5: strong disk I/O signals (rkb_s, r_s, r_await, io.util, io.w_s) with anomalies starting at ~2022-03-21 07:30:00 CST. Suggests heavy read activity and increased I/O wait.\n"
    "  * node-1: elevated system.cpu.iowait (severity_ratio ~7x relative to P95) with a few anomalous points — check for processes causing I/O wait on node-1.\n"
    "  * node-2: increased TCP retransmissions (system.net.tcp.retrans_segs) with anomaly_count > 0 — indicates possible network issues affecting node-2.\n"
    "  * node-3: elevated IO service time (svctm), io.util, and CPU user/pct_usage increases (severity_ratio ~2–4x) at mid-window times (around 07:43–07:49 CST).\n\n"
    "- Caveat: many series have global_P95 == 0 so the severity_ratio is inflated; treat those as I/O/network activity appearances rather than pure relative spikes. For actionable priority, focus on metrics with non-zero global_P95 and high absolute increases (e.g., iowait, retrans_segs, io.util, svctm).\n\n"
    "- Recommended next steps: investigate disk I/O saturation and queue lengths on node-4/node-5 (check disk device stats, IO queue, top I/O processes), inspect processes causing high iowait on node-1, examine network errors/retransmissions on node-2, and correlate node-3 CPU/IO spikes with workload or scheduled jobs. Also correlate with pod placements and high container memory/activity previously observed on node-6."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                     kpi_name  global_P95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  severity_ratio
0   node-4              system.io.rkb_s        0.00                      31             31    2022-03-21 07:30:00 CST                60.00    6.000000e+13
1   node-5              system.io.rkb_s        0.00                      31             31    2022-03-21 07:30:00 CST                48.00    4.800000e+13
2   node-4                system.io.r_s        0.00                      31             31    2022-03-21 07:30:00 CST                 4.00    4.000000e+12
3   node-5                system.io.r_s        0.00                      31             31    2022-03-21 07:30:00 CST                 2.50    2.500000e+12
4   node-5            system.io.r_await        0.00                      31             31    2022-03-21 07:30:00 CST                 0.50    5.000000e+11
5   node-4            system.io.r_await        0.00                      31             31    2022-03-21 07:30:00 CST                 0.38    3.800000e+11
6   node-1           system.io.avg_q_sz        0.00                      31             31    2022-03-21 07:30:00 CST                 0.01    1.000000e+10
7   node-2  system.net.tcp.retrans_segs        0.12                      31              4    2022-03-21 07:30:00 CST                 1.48    1.233333e+01
8   node-1            system.cpu.iowait        1.46                      31              3    2022-03-21 07:30:00 CST                10.38    7.109589e+00
9   node-3               system.io.util        1.40                      31              2    2022-03-21 07:45:00 CST                 6.30    4.500000e+00
10  node-3              system.io.svctm        0.40                      31              3    2022-03-21 07:43:00 CST                 1.53    3.825000e+00
11  node-3              system.cpu.user        1.79                      31              3    2022-03-21 07:49:00 CST                 6.17    3.446927e+00
12  node-4               system.io.util        0.05                      31              7    2022-03-21 07:35:00 CST                 0.15    3.000000e+00
13  node-2           system.io.avg_q_sz        0.05                      31              3    2022-03-21 07:38:00 CST                 0.15    3.000000e+00
14  node-4              system.cpu.user        1.26                      31              4    2022-03-21 07:41:00 CST                 3.09    2.452381e+00
15  node-2              system.io.svctm        0.41                      31              4    2022-03-21 07:36:00 CST                 0.97    2.365854e+00
16  node-2              system.cpu.user        1.63                      31              4    2022-03-21 07:43:00 CST                 3.66    2.245399e+00
17  node-2               system.io.util        1.60                      31              2    2022-03-21 07:39:00 CST                 3.40    2.125000e+00
18  node-3         system.cpu.pct_usage        3.96                      31              3    2022-03-21 07:49:00 CST                 7.97    2.012626e+00
19  node-4                system.io.w_s        0.50                      31              5    2022-03-21 07:35:00 CST                 1.00    2.000000e+00, 352, 311)```
```

In [ ]:
# In[10]:

import pandas as pd

# Prefer using the full per-series window aggregates if available (result_df), else fall back to filtered.
if 'result_df' in globals():
    per_series = result_df
elif 'filtered' in globals():
    per_series = filtered
else:
    raise RuntimeError("Expected per-series aggregates in 'result_df' or 'filtered' variables in the kernel.")

# 1) select series where kpi_name contains 'mesh.source' OR 'mesh.destination' (case-insensitive)
mask = per_series['kpi_name'].str.contains('mesh.source', case=False, na=False) | per_series['kpi_name'].str.contains('mesh.destination', case=False, na=False)
mesh_df = per_series[mask].copy()

# Ensure required columns exist (they should from prior steps)
cols_needed = ['cmdb_id', 'kpi_name', 'global_P95', 'total_points_in_window', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']
for c in cols_needed:
    if c not in mesh_df.columns:
        mesh_df[c] = None

# 2) Use existing per-series aggregates (mesh_df already contains them). If earliest_anomaly_timestamp is datetime, format it.
if pd.api.types.is_datetime64_any_dtype(mesh_df.get('earliest_anomaly_timestamp')):
    mesh_df['earliest_anomaly_timestamp'] = mesh_df['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# 3) Top 20 ordered by anomaly_count desc then max_value_in_window desc
mesh_top20 = mesh_df.sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False])[cols_needed].head(20).reset_index(drop=True)

# 4) counts: total distinct (cmdb_id,kpi_name) mesh series processed, and number with anomaly_count > 0
total_mesh_series = int(mesh_df.shape[0])
num_mesh_with_anom = int((mesh_df['anomaly_count'] > 0).sum())

# Keep mesh_df (full set) and mesh_top20 for follow-up steps in kernel
mesh_df = mesh_df
mesh_top20, total_mesh_series, num_mesh_with_anom

```
Out[10]:
```
Summary of Istio/mesh metric series (case-insensitive 'mesh.source' or 'mesh.destination'):

- Scope and counts:
  - Total mesh series processed: 1,660 distinct (cmdb_id, kpi_name).
  - Series with anomalies (value >= global_P95) in the incident window: 1,642.

- Top examples (from highest anomaly counts / values):
  - adservice2-0 — mesh.destination.unknown.adservice2.istio_resp...: global_P95 = 71,400, max = 74,225, anomaly_count = 31 (every minute in window).
  - adservice-0 — mesh.destination.unknown.adservice.istio_resp...: global_P95 = 71,400, max = 71,400, anomaly_count = 31.
  - adservice2-0 — mesh.destination.unknown.adservice2.istio_request...: global_P95 = 4,740, max = 4,937.5, anomaly_count = 31.
  - adservice-0 — similar mesh.destination request metrics also anomalous.
  - adservice2-0 — several mesh.source.* metrics (e.g., istio_tcp_connect/retrans related) show global_P95 = 0 but non-zero max values during the window and anomaly_count = 31.

- Pattern and interpretation:
  - Most mesh series are flagged for the entire window (anomaly_count = 31), which indicates sustained elevated mesh traffic/metrics during 07:30–08:00 CST rather than isolated spikes.
  - Some high-impact metrics (response/request counts between services) have large absolute values above historical P95 (e.g., tens of thousands), pointing to heavy request volume on meshed paths to adservice/adservice2.
  - Many series have global_P95 == 0 and small non-zero values in the window; these are flagged because historical baseline was zero. Treat those as "appearance of traffic/metrics" rather than necessarily critical issues.

- Recommended next steps:
  - Investigate increased request/response volumes on mesh.destination metrics for adservice/adservice2 (traffic surge or routing change).
  - Inspect mesh.source metrics showing TCP/connect/retrans activity (possible network issues or retries).
  - Correlate with frontend and log.row_count increases seen earlier to determine whether this is a system-wide traffic surge or an Istio/mesh-related degradation.

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id                                           kpi_name  global_P95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
0     adservice2-0  mesh.destination.unknown.adservice2.istio_resp...     71400.0                      31             31    2022-03-21 07:30:00 CST          74225.00000
1      adservice-0  mesh.destination.unknown.adservice.istio_respo...     71400.0                      31             31    2022-03-21 07:30:00 CST          71400.00000
2     adservice2-0  mesh.destination.unknown.adservice2.istio_requ...      4740.0                      31             31    2022-03-21 07:30:00 CST           4937.50000
3      adservice-0  mesh.destination.unknown.adservice.istio_reque...      4740.0                      31             31    2022-03-21 07:30:00 CST           4740.00000
4     adservice2-0  mesh.source.adservice2.basic-tidb.istio_tcp_se...         0.0                      31             31    2022-03-21 07:30:00 CST           1229.00000
5     adservice2-0  mesh.source.adservice2.basic-tidb.istio_tcp_re...         0.0                      31             31    2022-03-21 07:30:00 CST            654.00000
6      adservice-0  mesh.destination.unknown.adservice.istio_reque...        12.0                      31             31    2022-03-21 07:30:00 CST             12.00000
7     adservice2-0  mesh.destination.unknown.adservice2.istio_requ...        12.0                      31             31    2022-03-21 07:30:00 CST             12.00000
8     adservice2-0  mesh.source.adservice2.basic-tidb.istio_tcp_co...         0.0                      31             31    2022-03-21 07:30:00 CST              0.50000
9     adservice2-0  mesh.source.adservice2.basic-tidb.istio_tcp_co...         0.0                      31             31    2022-03-21 07:30:00 CST              0.50000
10  emailservice-1  mesh.source.emailservice.jaeger-collector.isti...         0.0                      31             31    2022-03-21 07:30:00 CST              0.00205
11  emailservice-2  mesh.source.emailservice.jaeger-collector.isti...         0.0                      31             31    2022-03-21 07:30:00 CST              0.00105
12     adservice-0  mesh.destination.frontend.adservice.istio_requ...         0.0                      31             31    2022-03-21 07:30:00 CST              0.00000
13     adservice-0  mesh.destination.frontend.adservice.istio_requ...         0.0                      31             31    2022-03-21 07:30:00 CST              0.00000
14     adservice-0  mesh.destination.frontend.adservice.istio_requ...         0.0                      31             31    2022-03-21 07:30:00 CST              0.00000
15     adservice-0  mesh.destination.frontend.adservice.istio_requ...         0.0                      31             31    2022-03-21 07:30:00 CST              0.00000
16     adservice-0  mesh.destination.frontend.adservice.istio_requ...         0.0                      31             31    2022-03-21 07:30:00 CST              0.00000
17     adservice-0  mesh.destination.frontend.adservice.istio_requ...         0.0                      31             31    2022-03-21 07:30:00 CST              0.00000
18     adservice-0  mesh.destination.frontend.adservice.istio_resp...         0.0                      31             31    2022-03-21 07:30:00 CST              0.00000
19     adservice-0  mesh.destination.frontend.adservice.istio_resp...         0.0                      31             31    2022-03-21 07:30:00 CST              0.00000, 1660, 1642)```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz

# reuse metric_df from kernel
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# ensure types
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# 1) select frontend/frontend2 rows and app.http or app.grpc KPIs (case-insensitive)
cmask = metric_df['cmdb_id'].astype(str).str.lower().str.startswith(('frontend','frontend2'))
kmask = metric_df['kpi_name'].astype(str).str.contains(r'app\.http|app\.grpc', case=False, na=False)
frontend_all = metric_df[cmask & kmask].copy()  # full-series for these frontend KPIs

# 2) compute global P95 per (cmdb_id, kpi_name) using the full frontend series (global thresholds)
frontend_p95 = (
    frontend_all
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_P95'})
)

# convert timestamps to tz-aware datetimes and filter to incident window (inclusive)
frontend_all['dt'] = pd.to_datetime(frontend_all['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start = pd.Timestamp('2022-03-21 07:30:00', tz=tz)
end = pd.Timestamp('2022-03-21 08:00:00', tz=tz)
frontend_window = frontend_all[(frontend_all['dt'] >= start) & (frontend_all['dt'] <= end)].copy()

# merge global P95 into the windowed frontend data
frontend_window = frontend_window.merge(frontend_p95, on=['cmdb_id', 'kpi_name'], how='left')

# flag anomalies where value >= global_P95
frontend_window['is_anom'] = frontend_window['value'] >= frontend_window['global_P95']

# aggregates per (cmdb_id, kpi_name) in-window
frontend_agg = (
    frontend_window
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        total_points_in_window=('value', 'size'),
        anomaly_count=('is_anom', 'sum'),
        max_value_in_window=('value', 'max')
    )
)

# earliest anomaly timestamp per series (min dt where is_anom True)
earliest_frontend_anom = (
    frontend_window.loc[frontend_window['is_anom']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['dt']
    .min()
    .rename(columns={'dt': 'earliest_anomaly_dt'})
)

# merge earliest timestamps and global_P95
frontend_result = frontend_agg.merge(earliest_frontend_anom, on=['cmdb_id', 'kpi_name'], how='left')
frontend_result = frontend_result.merge(frontend_p95, on=['cmdb_id', 'kpi_name'], how='left')

# format earliest_anomaly_timestamp as string in Asia/Shanghai; keep None where missing
frontend_result['earliest_anomaly_timestamp'] = frontend_result['earliest_anomaly_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
frontend_result['earliest_anomaly_timestamp'] = frontend_result['earliest_anomaly_timestamp'].where(frontend_result['earliest_anomaly_timestamp'].notna(), None)

# 3) compact table: top 20 ordered by anomaly_count desc then max_value_in_window desc
out_cols = ['cmdb_id', 'kpi_name', 'global_P95', 'total_points_in_window',
            'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']
frontend_top20 = frontend_result.sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False])[out_cols].head(20).reset_index(drop=True)

# 4) counts
num_frontend_series_processed = int(frontend_result.shape[0])
num_frontend_series_with_anom = int((frontend_result['anomaly_count'] > 0).sum())

# Keep frontend_p95 and frontend_window for later steps (they are assigned)
frontend_top20, num_frontend_series_processed, num_frontend_series_with_anom

```
Out[11]:
```
Summary of frontend request KPIs (incident window 2022-03-21 07:30–08:00 CST):

- Series examined: 4 frontend app.http/app.grpc series. All 4 showed anomaly_count > 0.

- Key results:
  - app.http.rr and app.http.sr — global P95 = 100; anomaly_count = 31 (every minute flagged) with earliest anomaly 2022-03-21 07:30:00 CST and max = 100. These are success/response-rate metrics equal to their historical P95 (i.e., stable at 100%).
  - app.http.count — global P95 = 140, anomaly_count = 2, earliest anomaly 2022-03-21 07:48:00 CST, max = 142. Only two minutes exceeded the P95 by a small amount.
  - app.http.mrt — global P95 ≈ 0.55s, anomaly_count = 4, earliest anomaly 2022-03-21 07:57:00 CST, max ≈ 1.13s (latency/mean response time spike late in the window).

- Interpretation:
  - There is no clear large traffic spike from frontend HTTP/grpc counts at 07:30:00 CST — app.http.count only exceeded its P95 briefly (2 minutes) and small margin. Success/response rates were simply at their P95 (100%) and were flagged by the numeric rule but are not a sign of surge.
  - The mesh and node I/O anomalies observed earlier are more consistent with sustained high throughput across many services (log.row_count and mesh.destination/request metrics were high) and with I/O/network activity appearing on nodes, rather than a sudden frontend-originated spike at 07:30.
  - The frontend did show a small number of latency anomalies later in the window (~07:57), which could be a symptom of downstream resource pressure.

- Recommendation:
  - Investigate sustained high traffic across the service mesh (mesh.destination/request metrics and log.row_count) and node I/O on nodes (node-4/5) rather than assuming a single abrupt surge from frontend(s) at 07:30.
  - Correlate per-minute frontend counts (logs) with mesh destination counts and node I/O timestamps; focus on the late-window latency spikes (~07:57) for possible downstream bottlenecks.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id        kpi_name  global_P95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
0  frontend     app.http.rr  100.000000                      31             31    2022-03-21 07:30:00 CST           100.000000
1  frontend     app.http.sr  100.000000                      31             31    2022-03-21 07:30:00 CST           100.000000
2  frontend    app.http.mrt    0.549454                      31              4    2022-03-21 07:57:00 CST             1.134435
3  frontend  app.http.count  140.000000                      31              2    2022-03-21 07:48:00 CST           142.000000, 4, 4)```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz

# Use previously computed per-series aggregates (prefer result_df, else filtered, else mesh_df)
if 'result_df' in globals():
    per_series = result_df
elif 'filtered' in globals():
    per_series = filtered
elif 'mesh_df' in globals():
    per_series = mesh_df
else:
    raise RuntimeError("Expected per-series aggregates in 'result_df', 'filtered', or 'mesh_df' variables.")

# Ensure earliest_anomaly_timestamp is present and consistently formatted (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'earliest_anomaly_timestamp' in per_series.columns and pd.api.types.is_datetime64_any_dtype(per_series['earliest_anomaly_timestamp']):
    per_series['earliest_anomaly_timestamp'] = per_series['earliest_anomaly_timestamp'].dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# 1) Select series where kpi_name contains both 'mesh.destination' AND 'adservice' (case-insensitive)
mask = per_series['kpi_name'].str.contains('mesh.destination', case=False, na=False) & per_series['kpi_name'].str.contains('adservice', case=False, na=False)
mesh_ad_df = per_series[mask].copy()

# Ensure required columns exist
needed = ['cmdb_id', 'kpi_name', 'global_P95', 'total_points_in_window', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']
for c in needed:
    if c not in mesh_ad_df.columns:
        mesh_ad_df[c] = None

# 2 & 3) Prepare compact table ordered by max_value_in_window desc, top 20
mesh_ad_df_top20 = mesh_ad_df.sort_values(by=['max_value_in_window'], ascending=False)[needed].head(20).reset_index(drop=True)

# 4) counts: total matching series and how many had anomaly_count > 0
total_mesh_ad_series = int(mesh_ad_df.shape[0])
num_mesh_ad_with_anom = int((mesh_ad_df['anomaly_count'] > 0).sum())

# Keep mesh_ad_df (full set) and mesh_ad_df_top20 in kernel for follow-up
mesh_ad_df = mesh_ad_df

# Return compact outputs
mesh_ad_df_top20, total_mesh_ad_series, num_mesh_ad_with_anom

```
Out[12]:
```
Summary of mesh.destination → adservice findings (plain English):

- Matches: 80 distinct (cmdb_id, kpi_name) series found where kpi_name contains both "mesh.destination" and "adservice".
- All matched series were flagged in the incident window: 80 series had anomaly_count > 0 (i.e., anomalies observed).

Top examples / patterns:
- adservice2-0 — mesh.destination.unknown.adservice2.istio_resp...: global_P95 = 71,400, max = 74,225, anomaly_count = 31 (every minute), earliest = 2022-03-21 07:30:00 CST.
- adservice-0 — mesh.destination.unknown.adservice.istio_resp...: global_P95 = 71,400, max = 71,400, anomaly_count = 31, earliest = 07:30:00 CST.
- adservice2-0/adservice-0 also show large mesh.destination request counts (global_P95 ~4,740 with max ~4,937.5 and anomaly_count = 31).
- Several frontend→adservice destination metrics show elevated values (e.g., global_P95 in the tens of thousands or thousands), with smaller groups of minutes peaking around 07:35 or later, but many series are anomalous for the whole window (07:30–08:00).

Interpretation:
- There was sustained, elevated mesh traffic directed at adservice and adservice2 starting at the window start (07:30:00 CST). Many destination/request and response-count metrics exceed their historical P95 for the entire 31-minute window.
- Some series have max == global_P95 (steady/high values), while others exceed P95 — both indicate sustained high throughput rather than single transient spikes.

Suggested next steps:
- Investigate traffic sources and routing to adservice/adservice2 (e.g., frontend or internal service calls) and whether a traffic surge or routing change occurred around 07:30.
- Correlate these high mesh.destination counts with the node I/O and container memory signals already observed (to identify whether the traffic caused the node I/O/memory issues or vice versa).

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name  global_P95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
0   adservice2-0  mesh.destination.unknown.adservice2.istio_resp...     71400.0                      31             31    2022-03-21 07:30:00 CST              74225.0
1    adservice-0  mesh.destination.unknown.adservice.istio_respo...     71400.0                      31             31    2022-03-21 07:30:00 CST              71400.0
2    adservice-0  mesh.destination.frontend.adservice.istio_resp...     41800.0                      31              3    2022-03-21 07:35:00 CST              42675.0
3    adservice-1  mesh.destination.frontend.adservice.istio_resp...     41500.0                      31              3    2022-03-21 07:35:00 CST              41950.0
4    adservice-2  mesh.destination.frontend.adservice.istio_resp...     40775.0                      31              1    2022-03-21 07:50:00 CST              41275.0
5    adservice-0  mesh.destination.frontend.adservice.istio_requ...     19090.0                      31              3    2022-03-21 07:35:00 CST              19407.5
6    adservice-1  mesh.destination.frontend.adservice.istio_requ...     18797.5                      31              4    2022-03-21 07:35:00 CST              19050.0
7    adservice-2  mesh.destination.frontend.adservice.istio_requ...     18692.5                      31              1    2022-03-21 07:50:00 CST              18692.5
8   adservice2-0  mesh.destination.unknown.adservice2.istio_requ...      4740.0                      31             31    2022-03-21 07:30:00 CST               4937.5
9    adservice-0  mesh.destination.unknown.adservice.istio_reque...      4740.0                      31             31    2022-03-21 07:30:00 CST               4740.0
10   adservice-0  mesh.destination.frontend.adservice.istio_requ...        30.0                      31              3    2022-03-21 07:35:00 CST                 30.0
11   adservice-0  mesh.destination.frontend.adservice.istio_requ...        30.0                      31              3    2022-03-21 07:35:00 CST                 30.0
12   adservice-1  mesh.destination.frontend.adservice.istio_requ...        29.5                      31              3    2022-03-21 07:31:00 CST                 30.0
13   adservice-0  mesh.destination.frontend.adservice.istio_resp...        30.0                      31              3    2022-03-21 07:35:00 CST                 30.0
14   adservice-1  mesh.destination.frontend.adservice.istio_resp...        29.5                      31              3    2022-03-21 07:31:00 CST                 30.0
15   adservice-1  mesh.destination.frontend.adservice.istio_requ...        29.5                      31              3    2022-03-21 07:31:00 CST                 30.0
16   adservice-2  mesh.destination.frontend.adservice.istio_requ...        29.0                      31              2    2022-03-21 07:37:00 CST                 29.0
17   adservice-2  mesh.destination.frontend.adservice.istio_resp...        29.0                      31              2    2022-03-21 07:37:00 CST                 29.0
18   adservice-2  mesh.destination.frontend.adservice.istio_requ...        29.0                      31              2    2022-03-21 07:37:00 CST                 29.0
19   adservice-0  mesh.destination.unknown.adservice.istio_reque...        12.0                      31             31    2022-03-21 07:30:00 CST                 12.0, 80, 80)```
```

In [ ]:
# In[13]:

import pandas as pd
import pytz

# reuse metric_df if present
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# ensure types
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# 1) select node-6 rows
node6_df = metric_df[metric_df['cmdb_id'] == 'node-6'].copy()

# 2) compute global P95 per kpi_name using the full node-6 series (global thresholds)
node6_p95 = (
    node6_df
    .groupby('kpi_name', as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_P95'})
)

# 3) filter to incident window and compute aggregates
node6_df['dt'] = pd.to_datetime(node6_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start = pd.Timestamp('2022-03-21 07:30:00', tz=tz)
end = pd.Timestamp('2022-03-21 08:00:00', tz=tz)
node6_window = node6_df[(node6_df['dt'] >= start) & (node6_df['dt'] <= end)].copy()

# merge global P95 into windowed data
node6_window = node6_window.merge(node6_p95, on='kpi_name', how='left')

# flag anomalies where value >= global_P95
node6_window['is_anom'] = node6_window['value'] >= node6_window['global_P95']

# aggregates per kpi_name in the window
node6_agg_basic = (
    node6_window
    .groupby('kpi_name', as_index=False)
    .agg(
        total_points_in_window=('value', 'size'),
        anomaly_count=('is_anom', 'sum'),
        max_value_in_window=('value', 'max')
    )
)

# earliest anomaly timestamp per kpi_name (first dt where is_anom True)
earliest_node6_anom = (
    node6_window.loc[node6_window['is_anom']]
    .groupby('kpi_name', as_index=False)['dt']
    .min()
    .rename(columns={'dt': 'earliest_anomaly_dt'})
)

# merge earliest timestamp and global_P95
node6_result = node6_agg_basic.merge(earliest_node6_anom, on='kpi_name', how='left')
node6_result = node6_result.merge(node6_p95, on='kpi_name', how='left')

# format earliest_anomaly_timestamp as string in Asia/Shanghai; keep None where missing
node6_result['earliest_anomaly_timestamp'] = node6_result['earliest_anomaly_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
node6_result['earliest_anomaly_timestamp'] = node6_result['earliest_anomaly_timestamp'].where(node6_result['earliest_anomaly_timestamp'].notna(), None)

# 4) prepare compact table ordered by anomaly_count desc then max_value_in_window desc, top 50
out_cols = ['kpi_name', 'global_P95', 'total_points_in_window', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']
node6_top50 = node6_result.sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False])[out_cols].head(50).reset_index(drop=True)

# 5) counts
total_kpis_node6 = int(node6_p95.shape[0])
num_kpis_with_anom = int((node6_result['anomaly_count'] > 0).sum())

# Keep node6_p95, node6_window, node6_result in kernel for follow-up steps (they are assigned).
# Display compact outputs
node6_top50, total_kpis_node6, num_kpis_with_anom

```
Out[13]:
```
summary = (
    "Node-6 KPI summary (incident window 2022-03-21 07:30–08:00 CST):\n\n"
    "- Scope: 59 distinct kpi_name series for node-6 were examined; 49 of these had anomaly_count > 0 in the window.\n\n"
    "- Top observations: many node-6 KPIs were anomalous for the full window (anomaly_count = 31, earliest anomaly = 2022-03-21 07:30:00 CST) — examples include:\n"
    "  • system.os.nofile.max (high file-descriptor limit-related values),\n"
    "  • system.mem.total (high memory totals),\n"
    "  • system.udp.connect.num and ping.can_connect (network/connectivity metrics),\n"
    "  • system.tcp.retrans_pct (TCP retransmission percentage), and several system.io and cpu metrics.\n\n"
    "- Timing/patterns: many series show sustained values at-or-above their historical P95 throughout the 31-minute window (starting 07:30), while some resource-usage metrics (e.g., memory pct, CPU user/system) spike later in the window (~07:49–08:00).\n\n"
    "- Interpretation and next steps:\n"
    "  • This points to broad node-level resource and network activity on node-6 (and related nodes) rather than a single transient spike.\n"
    "  • Prioritize investigating file-descriptor pressure, overall memory usage, network retransmissions, and disk I/O on node-6; correlate with the previously observed high mesh.destination/adservice traffic and container memory signals on node-6 to identify causal direction (traffic→I/O/memory pressure or resource problems→request issues).\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                      kpi_name    global_P95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
0         system.os.nofile.max  6.553600e+06                      31             31    2022-03-21 07:30:00 CST         6.553600e+06
1             system.mem.total  3.211700e+04                      31             31    2022-03-21 07:30:00 CST         3.211700e+04
2       system.udp.connect.num  1.100000e+01                      31             31    2022-03-21 07:30:00 CST         1.100000e+01
3             ping.can_connect  1.000000e+00                      31             31    2022-03-21 07:30:00 CST         1.000000e+00
4       system.tcp.retrans_pct  1.000000e-02                      31             31    2022-03-21 07:30:00 CST         1.000000e-02
..                         ...           ...                     ...            ...                        ...                  ...
45        system.mem.pct_usage  2.149000e+01                      31              1    2022-03-21 08:00:00 CST         2.421000e+01
46  system.mem.real.pct_useage  1.789000e+01                      31              1    2022-03-21 08:00:00 CST         2.074000e+01
47             system.cpu.user  9.050000e+00                      31              1    2022-03-21 08:00:00 CST         1.157000e+01
48           system.cpu.system  5.790000e+00                      31              1    2022-03-21 08:00:00 CST         7.720000e+00
49      system.fs.inodes.total  6.559784e+09                      31              0                       None         6.558499e+09

[50 rows x 6 columns], 59, 49)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[14]:

import pandas as pd
import pytz

# reuse metric_df from kernel
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# ensure types
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# substrings to search (case-insensitive)
patterns = [
    'oom', 'oom_kill', 'oom_killer', 'restart', 'restarts',
    'container_restart', 'process.exit', 'process.terminated',
    'container.state', 'container.uptime'
]

# build mask for kpi_name containing any pattern (case-insensitive)
kname_lower = metric_df['kpi_name'].astype(str).str.lower()
mask = False
for p in patterns:
    mask = mask | kname_lower.str.contains(p, na=False)

# matching full-series dataframe
matching_full = metric_df[mask].copy()

# 1) compute global P95 per (cmdb_id, kpi_name) using full series (before filtering)
matching_p95_df = (
    matching_full
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_P95'})
)

# 2) filter to incident window (inclusive)
matching_full['dt'] = pd.to_datetime(matching_full['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start = pd.Timestamp('2022-03-21 07:30:00', tz=tz)
end = pd.Timestamp('2022-03-21 08:00:00', tz=tz)
matching_window = matching_full[(matching_full['dt'] >= start) & (matching_full['dt'] <= end)].copy()

# 3) merge global_P95 into windowed rows, flag anomalies, and aggregate per series in-window
matching_window = matching_window.merge(matching_p95_df, on=['cmdb_id', 'kpi_name'], how='left')
matching_window['is_anom'] = matching_window['value'] >= matching_window['global_P95']

# aggregates for groups that have rows in window
window_agg = (
    matching_window
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        total_points_in_window=('value', 'size'),
        anomaly_count=('is_anom', 'sum'),
        max_value_in_window=('value', 'max')
    )
)

# earliest anomaly timestamp per group (where is_anom True)
earliest_anom = (
    matching_window.loc[matching_window['is_anom']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['dt']
    .min()
    .rename(columns={'dt': 'earliest_anomaly_dt'})
)

# 4) merge aggregates back into full list of matching series (so series with 0 points in window are included)
match_final = matching_p95_df.merge(window_agg, on=['cmdb_id', 'kpi_name'], how='left')
match_final = match_final.merge(earliest_anom, on=['cmdb_id', 'kpi_name'], how='left')

# fill missing numeric aggregates for series with no points in window
match_final['total_points_in_window'] = match_final['total_points_in_window'].fillna(0).astype(int)
match_final['anomaly_count'] = match_final['anomaly_count'].fillna(0).astype(int)
match_final['max_value_in_window'] = match_final['max_value_in_window'].fillna(0.0)

# format earliest_anomaly_timestamp as string (Asia/Shanghai) or None
match_final['earliest_anomaly_timestamp'] = match_final['earliest_anomaly_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
match_final['earliest_anomaly_timestamp'] = match_final['earliest_anomaly_timestamp'].where(match_final['earliest_anomaly_timestamp'].notna(), None)

# select output columns and sort
out_cols = ['cmdb_id', 'kpi_name', 'global_P95', 'total_points_in_window', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']
match_final_sorted = match_final[out_cols].sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False])

# top 20 compact table
match_top20 = match_final_sorted.head(20).reset_index(drop=True)

# total number of matching series (based on full metric.csv scan)
total_matching_series = int(matching_p95_df.shape[0])

# keep intermediate results in kernel for later steps:
# matching_p95_df, matching_window, match_final are saved variables.

match_top20, total_matching_series

```
Out[14]:
```
summary = (
    "Scan result: No metric KPI names in metric.csv matched the restart/OOM/process-termination patterns ('oom', 'oom_kill', 'restart', 'process.exit', 'container.state', etc.).\n\n"
    "Details: total matching series = 0. No per-series anomalies to report for these patterns.\n\n"
    "Interpretation / next steps: metric CSV does not show direct indicators of container restarts or OOM kills. "
    "Check error logs, Kubernetes events, and pod-level restart counters (e.g., kubelet/kube-state-metrics or error_logs.csv/log.csv) for evidence of restarts or OOMs."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, kpi_name, global_P95, total_points_in_window, anomaly_count, earliest_anomaly_timestamp, max_value_in_window]
Index: [], 0)```
```